# 🧬 NGS Analysis: From Reads to Variants

---

## Learning Objectives

- Understand Next-Generation Sequencing data
- Work with FASTQ format
- Perform quality control (FastQC)
- Basic variant calling workflow

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## NGS Workflow Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         NGS ANALYSIS PIPELINE                           │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Raw Reads      Quality          Alignment         Variant             │
│   (FASTQ)     →  Control      →   (BAM)         →   Calling             │
│                  (FastQC)         (BWA/Bowtie)      (GATK/bcftools)     │
│                                                                         │
│  ┌─────────┐    ┌─────────┐     ┌─────────┐      ┌─────────┐           │
│  │@read1   │    │ Check:  │     │ Align   │      │  Find   │           │
│  │ACGTACGT │ →  │ Quality │  →  │ to ref  │  →   │  SNPs   │           │
│  │+        │    │ Length  │     │ genome  │      │ Indels  │           │
│  │IIIIIIII │    │ Adapters│     │         │      │         │           │
│  └─────────┘    └─────────┘     └─────────┘      └─────────┘           │
│                                                                         │
│                           Output: VCF file                              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## FASTQ Format

```
@SEQ_ID                              ← Header line (starts with @)
GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTT  ← Sequence
+                                    ← Separator (starts with +)
!''*((((***+))%%%++)(%%%%).1***-+*''))**55CCF>>>>>>CCCCCCC65  ← Quality scores

Quality Score Encoding (Phred+33, Illumina 1.8+):
┌─────────┬────────────┬────────────────────────────────┐
│ Char    │ Phred      │ Error Probability              │
├─────────┼────────────┼────────────────────────────────┤
│ !       │ 0          │ 100% (worst)                   │
│ '       │ 6          │ 25%                            │
│ 5       │ 20         │ 1%                             │
│ ?       │ 30         │ 0.1%                           │
│ I       │ 40         │ 0.01% (best)                   │
└─────────┴────────────┴────────────────────────────────┘

    Phred = -10 × log₁₀(P_error)
    
    ASCII character value - 33 = Phred score
    Example: 'I' = ASCII 73, Phred = 73-33 = 40
```

In [ ]:
import math

def phred_to_probability(phred_score):
    """
    Convert Phred quality score to error probability.
    
    Phred = -10 × log₁₀(P_error)
    P_error = 10^(-Phred/10)
    """
    return 10 ** (-phred_score / 10)

def ascii_to_phred(char, encoding=33):
    """
    Convert ASCII character to Phred score.
    
    Parameters:
        char: ASCII quality character
        encoding: Offset (33 for Sanger/Illumina 1.8+)
    """
    return ord(char) - encoding

def parse_fastq(fastq_file):
    """
    Parse FASTQ file.
    
    Yields:
        Tuples of (header, sequence, quality_string)
    """
    with open(fastq_file, 'r') as f:
        while True:
            header = f.readline().strip()
            if not header:
                break
            sequence = f.readline().strip()
            plus = f.readline().strip()  # Skip + line
            quality = f.readline().strip()
            
            yield (header[1:], sequence, quality)  # Remove @ from header

def calculate_average_quality(quality_string):
    """
    Calculate average Phred quality of a read.
    """
    phred_scores = [ascii_to_phred(c) for c in quality_string]
    return sum(phred_scores) / len(phred_scores)

# Example
example_quality = "IIIIIIIIIIIIIIIIIIIIIIII"
avg_q = calculate_average_quality(example_quality)
print(f"Quality string: {example_quality}")
print(f"Average Phred score: {avg_q:.1f}")
print(f"Error probability: {phred_to_probability(avg_q):.4%}")

---

## Quality Control with FastQC

FastQC checks:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    FASTQC QUALITY METRICS                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  1. Per-base sequence quality                                           │
│     ┌──────────────────────────────┐                                    │
│     │▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░░░░░░ │ ← Quality drops at end            │
│     │ Good    │    Poor           │                                    │
│     └──────────────────────────────┘                                    │
│                                                                         │
│  2. Per-sequence quality scores                                         │
│     Most reads should have high average quality                         │
│                                                                         │
│  3. GC content distribution                                             │
│     Should match expected genome GC%                                    │
│                                                                         │
│  4. Sequence duplication levels                                         │
│     High duplication may indicate PCR bias                              │
│                                                                         │
│  5. Overrepresented sequences                                           │
│     Adapter contamination or rRNA                                       │
│                                                                         │
│  6. Adapter content                                                     │
│     Need trimming if adapters present                                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def basic_qc_report(fastq_file):
    """
    Generate basic QC statistics for FASTQ file.
    """
    total_reads = 0
    total_bases = 0
    quality_sum = 0
    gc_count = 0
    lengths = []
    
    for header, seq, qual in parse_fastq(fastq_file):
        total_reads += 1
        total_bases += len(seq)
        lengths.append(len(seq))
        
        # Quality
        for q_char in qual:
            quality_sum += ascii_to_phred(q_char)
        
        # GC content
        gc_count += seq.count('G') + seq.count('C')
    
    report = {
        'total_reads': total_reads,
        'total_bases': total_bases,
        'avg_length': sum(lengths) / len(lengths) if lengths else 0,
        'avg_quality': quality_sum / total_bases if total_bases else 0,
        'gc_content': gc_count / total_bases * 100 if total_bases else 0
    }
    
    return report

print("QC report function defined.")
print("Usage: report = basic_qc_report('reads.fastq')")

---

## VCF Format for Variants

```
##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
#CHROM  POS     ID      REF     ALT     QUAL    FILTER  INFO
chr1    12345   rs123   A       G       100     PASS    DP=50
chr1    12400   .       CT      C       80      PASS    DP=45

┌─────────────────────────────────────────────────────────────────────────┐
│                        VCF COLUMN MEANINGS                              │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  CHROM  - Chromosome                                                    │
│  POS    - 1-based position                                              │
│  ID     - Variant identifier (rsID or .)                               │
│  REF    - Reference allele                                              │
│  ALT    - Alternate allele(s)                                           │
│  QUAL   - Phred-scaled quality score                                    │
│  FILTER - PASS or filter reason                                         │
│  INFO   - Additional information (key=value pairs)                      │
│                                                                         │
│  Variant types:                                                         │
│  - SNP: A → G (single nucleotide)                                       │
│  - Insertion: A → AT (add bases)                                        │
│  - Deletion: AT → A (remove bases)                                      │
│  - MNP: AT → GC (multiple nucleotides)                                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_vcf(vcf_file):
    """
    Parse VCF file.
    
    Returns:
        List of variant dictionaries
    """
    variants = []
    
    with open(vcf_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            fields = line.strip().split('\t')
            if len(fields) >= 8:
                variant = {
                    'chrom': fields[0],
                    'pos': int(fields[1]),
                    'id': fields[2],
                    'ref': fields[3],
                    'alt': fields[4],
                    'qual': float(fields[5]) if fields[5] != '.' else None,
                    'filter': fields[6],
                    'info': dict(item.split('=') for item in fields[7].split(';') if '=' in item)
                }
                variants.append(variant)
    
    return variants

def classify_variant(ref, alt):
    """
    Classify variant type.
    """
    if len(ref) == 1 and len(alt) == 1:
        return 'SNP'
    elif len(ref) < len(alt):
        return 'insertion'
    elif len(ref) > len(alt):
        return 'deletion'
    else:
        return 'MNP'

# Example
print("Variant classification:")
print(f"  A → G: {classify_variant('A', 'G')}")
print(f"  A → AT: {classify_variant('A', 'AT')}")
print(f"  AT → A: {classify_variant('AT', 'A')}")
print(f"  AT → GC: {classify_variant('AT', 'GC')}")

---

## 🏋️ Exercises

### Exercise 1: Quality Filtering
Write a script to filter reads by quality score.

### Exercise 2: GC Content Analysis
Calculate GC content distribution across all reads.

### Exercise 3: Variant Statistics
Parse a VCF file and count variant types by chromosome.

---

## 📚 Resources

- [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/)
- [BWA](http://bio-bwa.sourceforge.net/)
- [GATK](https://gatk.broadinstitute.org/)
- [VCF Specification](https://samtools.github.io/hts-specs/VCFv4.2.pdf)